**A taste of remote sensing big data**

# **To test Dask’s multi-threading/multi-processing**

**Load the necessary Python Packages**

netCDF4 needs to be installed to the environment

In [23]:
from dask.distributed import Client 
import dask
from osgeo import gdal
%matplotlib inline
import os, glob, pathlib
import xarray as xr

**Set path to data**

In [24]:
DATA_PATH = pathlib.Path().cwd().parent / "data"
DATA_PATH

PosixPath('/Users/simenkrogstie/Documents/Programming/H26/GMFM350/notebooks/data')

**Start a Dask Dashboard to use distributed scheduler**

In [25]:
# Start a local Dask cluster
client = Client()

print("Dask cluster:", client)

/Users/simenkrogstie/Documents/Programming/H26/GMFM350/.venv/lib/python3.12/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 57943 instead
  warnings.warn(


Dask cluster: <Client: 'tcp://127.0.0.1:57944' processes=5 threads=10, memory=16.00 GiB>


In [26]:
client.scheduler_info()["workers"]

{'tcp://127.0.0.1:57957': {'type': 'Worker',
  'id': 0,
  'host': '127.0.0.1',
  'resources': {},
  'local_directory': '/var/folders/t9/z7ntnjlj50q8tsg9qqq5lxgc0000gn/T/dask-scratch-space/worker-qc4gvv4n',
  'name': 0,
  'nthreads': 2,
  'memory_limit': 3435973836,
  'last_seen': 1790757768.036144,
  'services': {'dashboard': 57958},
  'metrics': {'task_counts': {},
   'bandwidth': {'total': 100000000, 'workers': {}, 'types': {}},
   'digests_total_since_heartbeat': {},
   'managed_bytes': 0,
   'spilled_bytes': {'memory': 0, 'disk': 0},
   'transfer': {'incoming_bytes': 0,
    'incoming_count': 0,
    'incoming_count_total': 0,
    'outgoing_bytes': 0,
    'outgoing_count': 0,
    'outgoing_count_total': 0},
   'event_loop_interval': 0.02,
   'cpu': 0.0,
   'memory': 64520192,
   'time': 1790757768.020844,
   'host_net_io': {'read_bps': 0.0, 'write_bps': 0.0},
   'host_disk_io': {'read_bps': 0.0, 'write_bps': 0.0},
   'num_fds': 22},
  'status': 'running',
  'nanny': 'tcp://127.0.0.1:

In [27]:
print("Number of workers:", len(client.scheduler_info()["workers"]))

Number of workers: 5


---------------------

**Load data**

In [28]:
# Change current working directory to specified path
os.chdir(DATA_PATH)
# Display updated work directory path
os.getcwd()

'/Users/simenkrogstie/Documents/Programming/H26/GMFM350/notebooks/data'

**Get a list of the VRT files in the directory**

In [29]:
# os.path.join() finds all vrt files the directory. 
# If you want to search subdirectories recursively, add the recursive=True argument and use ** in your pattern
vrt_files = glob.glob(os.path.join(DATA_PATH, "**", '*.vrt'), recursive=True)
vrt_files

['/Users/simenkrogstie/Documents/Programming/H26/GMFM350/notebooks/data/vnir_bands.vrt',
 '/Users/simenkrogstie/Documents/Programming/H26/GMFM350/notebooks/data/all_bands.vrt',
 '/Users/simenkrogstie/Documents/Programming/H26/GMFM350/notebooks/data/T32VNM_20220517T103631.vrt']

**Read the large image file as a dataset**

Use the VRT file you created last week.

In [30]:
# Define correct vrt file
data = xr.open_dataset(vrt_files[2])

**Explore the dataset**

Varibales contain dataarrays among others. Dataarrays are N-dimensional numerical data such as images. 'band_data' is the default name for image data read typically from tif, jp2, and similar file formats.

In [31]:
# Explore the image dataarray
data['band_data']

<xarray.DataArray 'band_data' (band: 11, y: 10980, x: 10980)> Size: 5GB
[1326164400 values with dtype=float32]
Coordinates:
  * band         (band) int64 88B 1 2 3 4 5 6 7 8 9 10 11
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B ...
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        10720
    STATISTICS_MEAN:           1314.6136508918
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         364.42731648554
    STATISTICS_VALID_PERCENT:  100

**Chunking the data**

Large datasets can be challenging to fit into memory. Chunking the data into smaller pieces can help with this. The .chunk() method in Xarray breaks large arrays into smaller pieces called chunks for lazy, parallel processing using Dask

This data can be chunked along the 'band', 'x', and 'y' dimensions. Meaning we only call the defined number of bands and pixels into memory at the time.

In [32]:
# Here, we are chunking the data to 1 band and 512 x 512 pixels.
data_chunked = xr.open_dataset(vrt_files[2], chunks={'band': 1, 'x': 512, 'y': 512})
data_chunked

<xarray.Dataset> Size: 5GB
Dimensions:      (band: 11, x: 10980, y: 10980)
Coordinates:
  * band         (band) int64 88B 1 2 3 4 5 6 7 8 9 10 11
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (band, y, x) float32 5GB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>

**Explore the chunked dataset**

Now we get information both about the whole dataset and the chunks. Pay particular attention to how much less memory a chunk takes up compared to the entire dataset.

In [33]:
# Explore the image dataarray
band_data = data_chunked['band_data']
band_data

<xarray.DataArray 'band_data' (band: 11, y: 10980, x: 10980)> Size: 5GB
dask.array<open_dataset-band_data, shape=(11, 10980, 10980), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 88B 1 2 3 4 5 6 7 8 9 10 11
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B ...
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        10720
    STATISTICS_MEAN:           1314.6136508918
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         364.42731648554
    STATISTICS_VALID_PERCENT:  100

**Image statistics when the data is chunked**

Dask is lazy. This means that when you define operations on a dataset, Dask usually does not execute them immediately. Instead, it builds a task graph, which can be thought of as a blueprint describing all the calculations that need to be performed.

When a result is actually requested, for example by calling .compute(), saving to disk, or plotting data, Dask executes the graph. Because it has access to the entire workflow beforehand, it can optimize the calculations and distribute work across multiple cores or workers.

In the cells below, different statistics and computations are defined. Because the data are backed by Dask arrays, the operations are not executed immediately. Instead, Dask creates a task graph describing how the result can be calculated. The output shown is therefore not the final numerical result. Instead, it contains information about the future computation, such as the shape of the result, the chunk structure, memory requirements, and the number of tasks in the graph.

In [34]:
# Median values along the dimension of the bands. The output for each pixel would be the median value of all the bands.
band_data.median(dim='band')

<xarray.DataArray 'band_data' (y: 10980, x: 10980)> Size: 482MB
dask.array<nanmedian, shape=(10980, 10980), dtype=float32, chunksize=(1536, 1536), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B ...
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        10720
    STATISTICS_MEAN:           1314.6136508918
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         364.42731648554
    STATISTICS_VALID_PERCENT:  100

In [35]:
# Mean values along the dimension of the bands. The output for each pixel would be the mean value of all the bands.
band_data.mean(dim='band')

<xarray.DataArray 'band_data' (y: 10980, x: 10980)> Size: 482MB
dask.array<mean_agg-aggregate, shape=(10980, 10980), dtype=float32, chunksize=(512, 512), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B ...
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        10720
    STATISTICS_MEAN:           1314.6136508918
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         364.42731648554
    STATISTICS_VALID_PERCENT:  100

In [36]:
# NDVI: the output for each pixel would be the NDVI value calculated from the NIR and red bands
NDVI = (band_data[6] - band_data[2]) / (band_data[6] + band_data[2])
NDVI

<xarray.DataArray 'band_data' (y: 10980, x: 10980)> Size: 482MB
dask.array<truediv, shape=(10980, 10980), dtype=float32, chunksize=(512, 512), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B ...
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        10720
    STATISTICS_MEAN:           1314.6136508918
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         364.42731648554
    STATISTICS_VALID_PERCENT:  100

**Comparing processing without and with chunks**


Before running the next cells, open the localhost dashbard with the link bellow. Here you can can observ the parallel execution, in addition to seeing details and stats regarding the process when Dask is used

http://localhost:8787/

In [37]:
# Without chunks - "regular" Xarray/NumPy calculation
%time data['band_data'].mean(dim="band")

CPU times: user 12.1 s, sys: 6.39 s, total: 18.5 s
Wall time: 22.7 s


<xarray.DataArray 'band_data' (y: 10980, x: 10980)> Size: 482MB
array([[2573.0908, 2575.6365, 2615.5454, ..., 2526.818 , 2543.3635,
        2553.2727],
       [2647.5454, 2670.818 , 2632.2727, ..., 2548.2727, 2545.7273,
        2551.818 ],
       [2644.0908, 2639.3635, 2610.7273, ..., 2522.    , 2446.3635,
        2413.182 ],
       ...,
       [2292.7273, 2314.3635, 2382.0908, ..., 2709.9092, 2681.6365,
        2605.6365],
       [2302.182 , 2334.4546, 2340.7273, ..., 2747.5454, 2721.3635,
        2631.4546],
       [2296.2727, 2331.7273, 2345.2727, ..., 2737.4546, 2697.182 ,
        2658.7273]], shape=(10980, 10980), dtype=float32)
Coordinates:
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B ...
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        10720
    STATISTICS_MEAN:           1314.6136508918
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         364.42731648554
    STATISTICS_VALID_PERCENT:  100

First in the cell bellow the task graph is executed. Dask must now:
1. Read all chunks from the source data.
2. Calculate mean value for each chunk.
3. Combine all the chunks to one mean result.

In [38]:
# Dask executes chunks sequentially - chunks are processed one after another.
with dask.config.set(scheduler="single-threaded"):
    %time band_data.mean(dim='band').compute()
    # Note that we need the .compute() to tell Dask that we want to compute something

CPU times: user 12.9 s, sys: 915 ms, total: 13.8 s
Wall time: 16.4 s


In [39]:
# Dask executes chunks in parallel (uses the distributed scheduler) - chuncks areprocess simultaneous
%time band_data.mean(dim='band').compute()

CPU times: user 3.48 s, sys: 857 ms, total: 4.33 s
Wall time: 17.4 s


<xarray.DataArray 'band_data' (y: 10980, x: 10980)> Size: 482MB
array([[2573.0908, 2575.6365, 2615.5454, ..., 2526.818 , 2543.3635,
        2553.2727],
       [2647.5454, 2670.818 , 2632.2727, ..., 2548.2727, 2545.7273,
        2551.818 ],
       [2644.0908, 2639.3635, 2610.7273, ..., 2522.    , 2446.3635,
        2413.182 ],
       ...,
       [2292.7273, 2314.3635, 2382.0908, ..., 2709.9092, 2681.6365,
        2605.6365],
       [2302.182 , 2334.4546, 2340.7273, ..., 2747.5454, 2721.3635,
        2631.4546],
       [2296.2727, 2331.7273, 2345.2727, ..., 2737.4546, 2697.182 ,
        2658.7273]], shape=(10980, 10980), dtype=float32)
Coordinates:
  * y            (y) float64 88kB 6.7e+06 6.7e+06 6.7e+06 ... 6.59e+06 6.59e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B 0
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        10720
    STATISTICS_MEAN:           1314.6136508918
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         364.42731648554
    STATISTICS_VALID_PERCENT:  100

See how the different set ups give different results

**Preparing for computation**

In [40]:
data.rio.crs

CRS.from_wkt('PROJCS["WGS 84 / UTM zone 32N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",9],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32632"]]')

In [41]:
data.rio.crs.to_epsg()

32632

In [42]:
epsg = data.rio.crs.to_epsg()
# Convert the NDVI DataArray to a Dataset, with name "ndvi", and write the CRS information
NDVI = NDVI.to_dataset(name="ndvi")
NDVI = NDVI.rio.write_crs(f"EPSG:{epsg}")

# Define how the file should be encoded with chunking and zlib compression
encoding = {
    "ndvi": {
        "zlib": True,
        "complevel": 9, # 1 (fastest) to 9 (smallest)
        "chunksizes": ( 512, 512)
    }
}

**Writing a large result to disk**

Before running the next cell, open the localhost dashbard again: http://localhost:8787/

First here the NDVI task graph is executed. Dask must now:
1. Read all chunks from the source data.
2. Calculate NDVI for each chunk.
3. Compress the output.
4. Write the NetCDF file.

In [43]:
# Save the compressed NDVI data to a NetCDF file
NDVI.to_netcdf('ndvicompressed.nc', format="NETCDF4", engine="netcdf4", encoding=encoding)#rio.flip_axis("y")

/Users/simenkrogstie/Documents/Programming/H26/GMFM350/.venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/Users/simenkrogstie/Documents/Programming/H26/GMFM350/.venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/Users/simenkrogstie/Documents/Programming/H26/GMFM350/.venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/Users/simenkrogstie/Documents/Programming/H26/GMFM350/.venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/Users/simenkrogstie/Documents/Programming/H26/GMFM350/.venv/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


In [44]:
# Restart the Dask client to clear any cached data and free up memory
client.restart()

2026-09-30 10:49:51,352 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
2026-09-30 10:49:51,382 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
2026-09-30 10:49:51,383 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
2026-09-30 10:49:51,399 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
2026-09-30 10:49:51,410 - distributed.nanny - WARNING - Restarting worker (status=Status.running)


----
# **Assignement for this week**

Use what you have learned so far to:
  
- compute the NDVI, EVI, SAVI, NDWI and NDBI of an image of your choice
- compute thier correlations and histograms
- plot the results, i.e. indices, correlation and histogram, and submit as a pdf file


A notebook can easily be exported as a pdf file. In VS Code, press the 'export' function in the main toolbar of the nootebook (it is often hiden under the three dots). Chose PDF and define the placement and name. Note the `TeX` package must be installed in the environment for it to work. 

Sources to find satellite data:
- United States Geological Survey (USGS) Earth Explorer (Landsat)
- Copernicus Data Download (Sentinel)

Note that Copernicus gives you a safe folder with multiple folders and much minformation. The "most" relevant for you are:
- The jpg image, that gives a quick look of the downloaded image
- The IMG_DATA under the GRANULE folder. This is where the bands are stored